# # Fraud Detection - EDA & Data Quality Checks
# 
# This notebook performs Exploratory Data Analysis on `raw_insertions.parquet`.
# Goals:
# 1. Check data quality (nulls, types).
# 2. Analyze field coverage.
# 3. Explore fraud distribution and rates.
# 4. Analyze fraud label maturity (delay).

In [ ]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

# Configure Logic
DATA_PATH = "artifacts/raw_insertions.parquet"

# ## 1. Load Data & Basic Info

In [ ]:
print(f"Loading data from {DATA_PATH}...")
df = pl.read_parquet(DATA_PATH)
print(f"Shape: {df.shape}")
print("-" * 50)
print("Schema:")
print(df.schema)

# ## 2. Data Quality & Coverage

In [ ]:
# Check Nulls
null_counts = df.null_count()
total_rows = df.height

print("\nMissing Values & Coverage:")
coverage_stats = []
for col in df.columns:
    nulls = null_counts[col][0]
    pct_missing = (nulls / total_rows) * 100
    coverage = 100 - pct_missing
    dtype = str(df.schema[col])
    
    coverage_stats.append({
        "column": col,
        "dtype": dtype,
        "nulls": nulls,
        "pct_missing": round(pct_missing, 2),
        "coverage": round(coverage, 2)
    })

# Convert to simple dataframe for display
df_coverage = pl.DataFrame(coverage_stats).sort("pct_missing", descending=True)
print(df_coverage)

# Check Empty Strings (for string cols)
print("\nEmpty String Counts (for string columns):")
for col in df.select(pl.col(pl.Utf8)).columns:
    empty_count = df.filter(pl.col(col) == "").height
    if empty_count > 0:
        print(f"{col}: {empty_count} ({round(empty_count/total_rows*100, 2)}%)")

# ## 3. Type Distribution & Unique Values

In [ ]:
print("\nUnique Value Counts:")
for col in df.columns:
    # Cap generic printing
    n_unique = df.select(pl.col(col).n_unique()).item()
    print(f"{col}: {n_unique} unique")

# ## 4. Fraud Distribution

In [ ]:
# Target distribution
# Assuming 'is_fraud' is the target. If not, we look for status transitions or similar.
# The user mentioned 'raw_insertions'. Does it have labels?
# Typically 'raw_insertions' might need joining with 'status_history'.
# Let's check if 'is_fraud' exists. Not in schema?
# If missing, we might need to derive it or finding 'state' column.
# Let's inspect columns to find target.
# Look for specific columns based on previous run
TARGET_COL = "fraud_flag" 
time_col = "submission_at"

if TARGET_COL not in df.columns:
    print(f"Target '{TARGET_COL}' still not found. Available columns with 'fraud':")
    print([c for c in df.columns if "fraud" in c.lower()])
else:
    print(f"\nFraud Rate ({TARGET_COL}):")
    # Check values
    print(df[TARGET_COL].value_counts())
    
    # Check if boolean or int or string
    dtypes = df.schema[TARGET_COL]
    print(f"Target Type: {dtypes}")

# Time Analysis
if time_col in df.columns:
    # Ensure datetime
    # df = df.with_columns(pl.col(time_col).str.to_datetime()) # It might be string?
    
    min_date = df[time_col].min()
    max_date = df[time_col].max()
    print(f"\nTime Range: {min_date} to {max_date}")
    
    # Analyze daily volume
    # Cast to date if needed
    daily_vol = (
        df.group_by(pl.col(time_col).dt.date())
        .count()
        .sort(time_col)
    )
    
    # Plotting code remains similar...
    
# Feature Discovery (Bedrooms, Price, Area)
print("\nFeature Discovery:")
keywords = ["price", "area", "room", "bed", "bath", "m2", "size"]
for k in keywords:
    matches = [c for c in df.columns if k in c.lower()]
    print(f"Columns matching '{k}': {len(matches)}")
    if len(matches) < 10:
        print(matches)
    else:
        print(matches[:5] + ["..."])